In [1]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [2]:
model = pe.ConcreteModel()

Sets

In [4]:
model.product_types = pe.Set(initialize = ["A","B","C"])
model.factories = pe.Set(initialize = ["F1","F2","F3","F4","F5","F6",])

Parameters

In [5]:
capacity_dict = {
    "F1":550,
    "F2":700,
    "F3":1100,
    "F4":350,
    "F5":400,
    "F6":450
}
model.capacity_of_factory = pe.Param( model.factories, initialize=capacity_dict)


In [6]:
cost_dict = {
    ("A", "F1"): 25, ("A", "F2"): 30, ("A", "F3"): 26, ("A", "F4"): 34, ("A", "F5"): 32, ("A", "F6"): 30,
    ("B", "F1"): 30, ("B", "F2"): 32, ("B", "F3"): 34, ("B", "F4"): 35, ("B", "F5"): 38, ("B", "F6"): 40,
    ("C", "F1"): 40, ("C", "F2"): 46, ("C", "F3"): 42, ("C", "F4"): 37, ("C", "F5"): 40, ("C", "F6"): 50
}

model.unitary_cost = pe.Param(model.product_types, model.factories, initialize=cost_dict)

In [7]:
sold_dict = {
    "A": 700,
    "B": 500,
    "C": 600
}

model.already_sold = pe.Param(model.product_types, initialize=sold_dict)

In [8]:
price_dict = {
    "A": 60,
    "B": 82.5,
    "C": 108
}

model.selling_price = pe.Param(model.product_types, initialize=price_dict)

Variables

In [9]:
model.product_quantity = pe.Var(model.product_types, model.factories, within = pe.NonNegativeReals)

Objective Function

In [10]:
def obj_rule(model):
    revenue = sum(model.selling_price[p] * model.product_quantity[p, f]
                  for p in model.product_types for f in model.factories)
    cost = sum(model.unitary_cost[p, f] * model.product_quantity[p, f]
               for p in model.product_types for f in model.factories)
    return revenue - cost

model.profit = pe.Objective(rule=obj_rule, sense=pe.maximize)

Constraints

In [11]:
def contract_rule(model, p):
    return sum(model.product_quantity[p,f] for f in model.factories) >= model.already_sold[p]

model.contract_constraint = pe.Constraint(model.product_types, rule=contract_rule)

In [12]:
def capacity_rule(model, f):
    return sum(model.product_quantity[p,f] for p in model.product_types) <= model.capacity_of_factory[f]

model.capacity_constraint = pe.Constraint(model.factories, rule=capacity_rule)

In [13]:
#solver = po.SolverFactory('glpk')
solver = po.SolverFactory('gurobi_direct', executable = r"C:\gurobi1303\win64\bin\gurobi_cl.exe")
results = solver.solve(model, tee=True) 

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 9 rows, 18 columns and 36 nonzeros (Max)
Model fingerprint: 0x5bbc731c
Model has 18 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+01, 7e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+02, 1e+03]

Presolve time: 0.01s
Presolved: 9 rows, 18 columns, 36 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    8.6200000e+32   1.800000e+31   8.620000e+02      0s
      10    2.0520000e+05   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.02 seconds (0.00 work units)
Optimal objective  2.052000000e+05


In [14]:
print(pe.value(model.profit))


205200.0


In [18]:
solver = po.SolverFactory("gurobi_direct")

results = solver.solve(model, tee=True)

print("Status:", results.solver.status)
print("Termination:", results.solver.termination_condition)
print("Profit:", pe.value(model.profit))

print("\nSolución:")
for p in model.product_types:
    for f in model.factories:
        quantity = pe.value(model.product_quantity[p, f])

        if quantity > 1e-6:
            print(f"{p} en {f}: {quantity:.2f}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 9 rows, 18 columns and 36 nonzeros (Max)
Model fingerprint: 0x5bbc731c
Model has 18 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+01, 7e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+02, 1e+03]

Presolve time: 0.00s
Presolved: 9 rows, 18 columns, 36 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    8.6200000e+32   1.800000e+31   8.620000e+02      0s
      10    2.0520000e+05   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.052000000e+05
Status: ok
Termination: optimal
Profit: 205200.0

Solución:
A en F2: 200.00
A en F3: 50.00
A en F6: 450.0